# ES 678: Advanced Environmental Machine Learning
## Exploratory Data Analysis & Time-Series Diagnostics: Anand Vihar, Delhi

This notebook conducts preliminary exploratory data analysis (EDA), stationarity audits, autocorrelation profiling, and correlation analysis for 24-hour ahead AQI forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from src.data_loader import load_and_validate_data, audit_dataset

df = load_and_validate_data("data/raw/station_hourly.csv", synthetic_backup=True)
print("Shape:", df.shape)
df.head()

### 1. Dataset Audit & Extreme Event Prevalence

In [ ]:
audit = audit_dataset(df)
print("Date Range:", audit["date_range"])
print("\n--- CPCB Category Breakdown ---")
for cat, pct in audit["aqi_category_pct"].items():
    print(f"{cat:25s}: {pct:6.2f}% ({audit['aqi_category_counts'][cat]} hours)")

### 2. Time-Series Visualization & Seasonal Smog Inversion

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
df["AQI"].plot(ax=axes[0], color="crimson", lw=1.2, label="Hourly AQI")
axes[0].axhline(300, color="orange", ls="--", label="CPCB Very Poor (300)")
axes[0].axhline(400, color="darkred", ls=":", label="CPCB Severe (400)")
axes[0].set_ylabel("AQI (0-500)")
axes[0].legend(loc="upper right")

df[["PM2.5", "PM10"]].plot(ax=axes[1], alpha=0.8, lw=1.0)
axes[1].set_ylabel("PM Concentration (ug/m3)")

df["Wind_Speed"].plot(ax=axes[2], color="teal", lw=1.0, label="Wind Speed (m/s)")
axes[2].axhline(1.0, color="black", ls="--", label="Stagnation Threshold (1.0 m/s)")
axes[2].set_ylabel("Wind Speed (m/s)")
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

### 3. Stationarity Testing (Augmented Dickey-Fuller Test)

In [ ]:
adf_res = adfuller(df["AQI"].dropna())
print("ADF Statistic:", adf_res[0])
print("p-value:", adf_res[1])
print("Critical Values:", adf_res[4])
if adf_res[1] < 0.05:
    print("Series is stationary at 95% confidence level.")
else:
    print("Series is non-stationary; differencing or seasonal conditioning required.")

### 4. Correlation Matrix: Pollutants vs. Micro-Meteorology

In [ ]:
num_cols = ["AQI", "PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "Temperature", "Relative_Humidity", "Wind_Speed", "Pressure"]
corr = df[num_cols].corr()
plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
plt.yticks(range(len(num_cols)), num_cols)
for i in range(len(num_cols)):
    for j in range(len(num_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")
plt.title("Cross-Correlation Matrix: Criteria Pollutants vs Surface Meteorology")
plt.tight_layout()
plt.show()